In [8]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

In [9]:
device = 0 if torch.backends.mps.is_available() else -1

In [10]:
classifier = pipeline(
    "text-classification", 
    model="bhadresh-savani/distilbert-base-uncased-emotion", 
    top_k=None, # Zwróć wyniki dla wszystkich 6 emocji, a nie tylko dla tej najsilniejszej
    device=device 
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5845.09it/s]


In [11]:
def get_emotion_vector(text):
    clean_text = str(text)[:512]
    results = classifier(clean_text)
    return [res['score'] for res in results[0]]

In [12]:
df = pd.read_csv("../../../data/cleaned_data.csv")
features = []
for text in tqdm(df['cleaned_statement']):
    features.append(get_emotion_vector(text))

100%|██████████| 29995/29995 [04:23<00:00, 113.89it/s]


In [13]:
emotion_cols = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
features_df = pd.DataFrame(features, columns=emotion_cols)
features_df['label'] = df['label']

In [14]:
features_df.to_csv('../../../data/features_for_random_forest.csv', index=False)